In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score
from sklearn.metrics import root_mean_squared_error as RMSE

In [9]:
# Then in modeling notebook
train = pd.read_csv("C:\\Users\\Mr.Zabit\\Documents\\ml_projects\\bike_rental_project\\data\\processed\\train.csv")
test = pd.read_csv("C:\\Users\\Mr.Zabit\\Documents\\ml_projects\\bike_rental_project\\data\\processed\\test.csv")

# Linear Regression

In [10]:
# Define features
categorical_features = [
    'season', 'yr', 'mnth', 'hr',
    'holiday', 'weekday', 'workingday', 'weathersit'
]

numeric_features = ['temp', 'atemp', 'hum', 'windspeed']

X_train = train[categorical_features + numeric_features]
y_train = train['cnt']

X_test = test[categorical_features + numeric_features]
y_test = test['cnt']

In [11]:
# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Full pipeline
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('regression', LinearRegression())
])

In [12]:
# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)
print("MAE:", MAE(y_test.values, y_pred))
print("RMSE:", RMSE(y_test.values, y_pred))
print("R2:", r2_score(y_test.values, y_pred))

MAE: 100.90106335513255
RMSE: 135.11641735307256
R2: 0.6217424130268183


In [13]:
# Get feature names after one-hot encoding
feature_names = model.named_steps['preprocessing'].get_feature_names_out()

# Get coefficients
coefficients = model.named_steps['regression'].coef_

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
})

# Top positive
print("Top positive coefficients:")
print(coef_df.sort_values(by='coefficient', ascending=False).head(10))

# Top negative
print("\nTop negative coefficients:")
print(coef_df.sort_values(by='coefficient').head(10))


Top positive coefficients:
       feature  coefficient
31  cat__hr_17   335.251191
32  cat__hr_18   308.986357
22   cat__hr_8   275.451925
33  cat__hr_19   211.956735
30  cat__hr_16   196.593564
26  cat__hr_12   153.216141
27  cat__hr_13   150.580955
21   cat__hr_7   150.062900
23   cat__hr_9   147.166093
29  cat__hr_15   143.427942

Top negative coefficients:
              feature  coefficient
51           num__hum   -70.936211
47  cat__weathersit_3   -64.636297
48  cat__weathersit_4   -58.085137
52     num__windspeed   -35.584075
18          cat__hr_4   -34.624024
17          cat__hr_3   -31.813185
16          cat__hr_2   -22.201391
19          cat__hr_5   -19.912826
15          cat__hr_1   -14.733500
38     cat__holiday_1   -13.501357


## Questions

1. What is the linear model assuming about relationships between features and cnt?

The linear model thinks that each thing we use to predict (like hour, weather, or temperature) adds or reduces a fixed number of bike rentals.

2. Which coefficients match your intuition? Which do not? 

Hours increase rentals, bad weather and high humidity decrease rentals — these coefficients match intuition.